# Botify Recommender — превосходим LightFM бейзлайн
Цель: обучить модель, которая победит baseline по метрике **NDCG**.

## 1. Импорты и настройки

In [12]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
from pathlib import Path

DATA_DIR = Path("/Users/daniilkokorev/local-projects/RecSys_VK/recsys-course-spring-2026/botify/data")

train = pd.read_csv(DATA_DIR / "train.csv")
test  = pd.read_csv(DATA_DIR / "test.csv")

print("train:", train.shape)
print("test: ", test.shape)
train.head()

train: (2631427, 3)
test:  (657679, 2)


,user,track,time
0,0dffdaaffb460ead08234a09f625d3e8,4252,0.26
1,a5a049b5d915c48ee66fcf482818b30b,326,1.00
2,803deb36c0f2dae7306bec7ce5a0a2a0,9959,0.05
3,c7510a5422f0f55f02be7981b1158f27,4042,0.26
4,0181ae3f7155d55fa176840320e0c566,11371,0.04


## 2. EDA

In [13]:
print("=== Train ===")
print(f"Уникальных пользователей : {train['user'].nunique():,}")
print(f"Уникальных треков        : {train['track'].nunique():,}")
print(f"Всего взаимодействий     : {len(train):,}")
print(f"Sparsity                 : {1 - len(train) / (train['user'].nunique() * train['track'].nunique()):.4%}")
print()
print("=== Распределение time ===")
print(train['time'].describe())
print()
print(f"Доля полностью прослушанных (time=1.0) : {(train['time'] == 1.0).mean():.2%}")
print(f"Доля почти не прослушанных  (time<0.1) : {(train['time'] < 0.1).mean():.2%}")

=== Train ===
Уникальных пользователей : 10,000
Уникальных треков        : 16,198
Всего взаимодействий     : 2,631,427
Sparsity                 : 98.3755%

=== Распределение time ===
count    2.631427e+06
mean     4.300742e-01
std      3.319647e-01
min      0.000000e+00
25%      1.300000e-01
50%      3.700000e-01
75%      6.900000e-01
max      1.000000e+00
Name: time, dtype: float64

Доля полностью прослушанных (time=1.0) : 11.55%
Доля почти не прослушанных  (time<0.1) : 20.31%


In [14]:
# Активность пользователей
user_activity = train.groupby('user').size()
print("Взаимодействий на пользователя:")
print(user_activity.describe())

track_activity = train.groupby('track').size()
print("\nВзаимодействий на трек:")
print(track_activity.describe())

Взаимодействий на пользователя:
count    10000.000000
mean       263.142700
std         94.759176
min         79.000000
25%        196.000000
50%        239.000000
75%        307.000000
max        858.000000
dtype: float64

Взаимодействий на трек:
count    16198.000000
mean       162.453821
std         50.879936
min         51.000000
25%        128.000000
50%        156.000000
75%        187.000000
max        630.000000
dtype: float64


In [15]:
# Сколько пользователей из test есть в train?
train_users = set(train['user'].unique())
test_users  = set(test['user'].unique())
print(f"Пользователей в test  : {len(test_users):,}")
print(f"Из них есть в train   : {len(test_users & train_users):,} ({len(test_users & train_users)/len(test_users):.1%})")
print(f"Холодных пользователей: {len(test_users - train_users):,}")

train_tracks = set(train['track'].unique())
test_tracks  = set(test['track'].unique())
print(f"\nТреков в test         : {len(test_tracks):,}")
print(f"Из них есть в train   : {len(test_tracks & train_tracks):,} ({len(test_tracks & train_tracks)/len(test_tracks):.1%})")

Пользователей в test  : 9,993
Из них есть в train   : 9,993 (100.0%)
Холодных пользователей: 0

Треков в test         : 16,198
Из них есть в train   : 16,198 (100.0%)


## 3. Подготовка данных

In [ ]:
# Энкодим user и track в числовые индексы
all_users  = pd.concat([train['user'],  test['user']]).unique()
all_tracks = pd.concat([train['track'], test['track']]).unique()

user2idx  = {u: i for i, u in enumerate(all_users)}
track2idx = {t: i for i, t in enumerate(all_tracks)}
idx2track = {i: t for t, i in track2idx.items()}

n_users  = len(user2idx)
n_tracks = len(track2idx)

print(f"n_users={n_users:,}  n_tracks={n_tracks:,}")

train['uid'] = train['user'].map(user2idx)
train['tid'] = train['track'].map(track2idx)
test['uid']  = test['user'].map(user2idx)
test['tid']  = test['track'].map(track2idx)

# Метаданные треков: добавляем контентные признаки для LTR-моделей.
track_meta_raw = pd.read_json(DATA_DIR / "tracks.json", lines=True)
track_meta = track_meta_raw[track_meta_raw['track'].isin(all_tracks)].copy()
track_meta['tid'] = track_meta['track'].map(track2idx)
track_meta['artist_genres_str'] = track_meta['artist_genres'].map(lambda x: '|'.join(x) if isinstance(x, list) else str(x))
track_meta['genres_str'] = track_meta['genres'].map(lambda x: '|'.join(x) if isinstance(x, list) else str(x))
track_meta['mood_str'] = track_meta['mood'].map(lambda x: '|'.join(x) if isinstance(x, list) else str(x))

for col in ['artist', 'artist_country', 'artist_genre', 'artist_genres_str', 'genres_str', 'mood_str']:
    track_meta[f'{col}_code'] = pd.factorize(track_meta[col].fillna('__missing__'))[0].astype(np.int32)

track_meta_ltr = track_meta[[
    'tid', 'year', 'artist_fans', 'artist_id',
    'artist_code', 'artist_country_code', 'artist_genre_code',
    'artist_genres_str_code', 'genres_str_code', 'mood_str_code'
]].copy()

# Словарь tid -> metadata. Его использует make_ltr_features ниже.
track_meta_ltr = track_meta_ltr.drop_duplicates('tid').set_index('tid')
print(f"track metadata rows={len(track_meta_ltr):,}")

In [17]:
# Агрегируем повторные взаимодействия user-track.
# Важно: ниже будем много раз пересобирать confidence, поэтому сохраняем базовые статистики.
train_agg = train.groupby(['uid', 'tid']).agg(
    time_max=('time', 'max'),
    time_mean=('time', 'mean'),
    time_sum=('time', 'sum'),
    play_count=('time', 'count'),
).reset_index()

# Для совместимости с функциями NDCG: истинная релевантность трека = лучшее время прослушивания.
train_agg['time'] = train_agg['time_max']

print(f"После агрегации: {len(train_agg):,} user-track пар (было {len(train):,} событий)")
print(train_agg[['time_max', 'time_mean', 'time_sum', 'play_count']].describe())


def add_confidence(df, alpha=40, min_time=0.0, formula='max_log_count'):
    """Возвращает interactions с confidence для implicit ALS."""
    result = df[df['time_max'] >= min_time].copy()

    if formula == 'max_log_count':
        relevance = result['time_max'] * np.log1p(result['play_count'])
    elif formula == 'sum_time':
        relevance = np.log1p(result['time_sum'])
    elif formula == 'mix':
        relevance = (0.7 * result['time_max'] + 0.3 * result['time_mean']) * np.log1p(result['play_count'])
    else:
        raise ValueError(f"Unknown confidence formula: {formula}")

    result['confidence'] = 1.0 + alpha * relevance
    return result


def build_user_item(df):
    return sp.csr_matrix(
        (df['confidence'].values.astype(np.float32), (df['uid'].values, df['tid'].values)),
        shape=(n_users, n_tracks),
    )

# Дефолтная матрица оставлена для быстрых экспериментов ниже.
ALPHA = 40
train_agg_conf = add_confidence(train_agg, alpha=ALPHA, min_time=0.0, formula='max_log_count')
user_item = build_user_item(train_agg_conf)
print(f"User-item matrix: {user_item.shape}, nnz={user_item.nnz:,}")

После агрегации: 2,500,130 user-track пар (было 2,631,427 событий)
           time_max     time_mean      time_sum    play_count
count  2.500130e+06  2.500130e+06  2.500130e+06  2.500130e+06
mean   4.060193e-01  4.052260e-01  4.526600e-01  1.052516e+00
std    3.171834e-01  3.165971e-01  4.846015e-01  2.969663e-01
min    0.000000e+00  0.000000e+00  0.000000e+00  1.000000e+00
25%    1.200000e-01  1.200000e-01  1.200000e-01  1.000000e+00
50%    3.400000e-01  3.400000e-01  3.500000e-01  1.000000e+00
75%    6.400000e-01  6.400000e-01  6.500000e-01  1.000000e+00
max    1.000000e+00  1.000000e+00  1.200000e+01  1.200000e+01
User-item matrix: (10000, 16198), nnz=2,500,130


## 4. Валидация (hold-out)

In [18]:
def ndcg_at_k(actual_scores, predicted_scores, k=10):
    """NDCG@k для одного пользователя по кандидатам из test/validation."""
    actual_scores = np.asarray(actual_scores)
    predicted_scores = np.asarray(predicted_scores)

    if len(actual_scores) == 0:
        return 0.0

    k = min(k, len(actual_scores))
    top_k_pred = np.argsort(predicted_scores)[::-1][:k]
    top_k_ideal = np.argsort(actual_scores)[::-1][:k]

    def dcg(order, scores):
        gains = scores[order]
        discounts = np.log2(np.arange(2, len(order) + 2))
        return np.sum(gains / discounts)

    idcg = dcg(top_k_ideal, actual_scores)
    if idcg == 0:
        return 0.0
    return dcg(top_k_pred, actual_scores) / idcg


def evaluate_ndcg(test_df, score_col='pred', k=10):
    """Считает средний NDCG@k по пользователям."""
    ndcgs = []
    for _, grp in test_df.groupby('uid'):
        ndcgs.append(ndcg_at_k(grp['time'].values, grp[score_col].values, k=k))
    return float(np.mean(ndcgs))


def normalize_per_user(df, score_values, score_col):
    """Min-max нормализация скоринга внутри пользователя перед блендингом."""
    tmp = df[['uid']].copy()
    tmp[score_col] = score_values

    def norm(grp):
        mn, mx = grp[score_col].min(), grp[score_col].max()
        if mx > mn:
            grp[score_col] = (grp[score_col] - mn) / (mx - mn)
        else:
            grp[score_col] = 0.5
        return grp

    return tmp.groupby('uid', group_keys=False).apply(norm)[score_col].values


def als_candidate_scores(model, df):
    """Скор ALS для конкретных пар user-track из df."""
    return np.sum(
        model.user_factors[df['uid'].values] * model.item_factors[df['tid'].values],
        axis=1,
    )

print("Функции валидации готовы")

Функции валидации готовы


In [19]:
# Per-user hold-out: у каждого пользователя с >=2 треками оставляем часть взаимодействий в validation.
# Так validation ближе к лидерборду: ранжируем кандидаты внутри одного пользователя.
np.random.seed(42)

val_indices = []
for uid, grp in train_agg.groupby('uid'):
    if len(grp) < 2:
        continue
    n_val = max(1, int(round(0.2 * len(grp))))
    val_indices.extend(np.random.choice(grp.index.values, size=n_val, replace=False))

val_indices = np.array(val_indices)
val_mask = train_agg.index.isin(val_indices)

train_part_base = train_agg[~val_mask].copy()
val_part = train_agg[val_mask].copy()

print(f"Train part : {len(train_part_base):,}")
print(f"Val part   : {len(val_part):,}")
print(f"Val users  : {val_part['uid'].nunique():,}")

val_uids = val_part['uid'].values
val_tids = val_part['tid'].values

Train part : 2,000,085
Val part   : 500,045
Val users  : 10,000


## 5. Бейзлайн — Popularity

In [20]:
# Несколько popularity baseline. Их потом полезно блендить с ALS, особенно для холодных/редких пользователей.
def add_popularity_scores(train_df, eval_df):
    scored = eval_df.copy()
    pop_sum = train_df.groupby('tid')['time_sum'].sum()
    pop_mean = train_df.groupby('tid')['time_mean'].mean()
    pop_count = train_df.groupby('tid')['play_count'].sum()
    pop_good = train_df.assign(good=(train_df['time_max'] >= 0.7).astype(int)).groupby('tid')['good'].sum()

    scored['pop_sum'] = scored['tid'].map(pop_sum).fillna(0).values
    scored['pop_mean'] = scored['tid'].map(pop_mean).fillna(0).values
    scored['pop_count'] = scored['tid'].map(pop_count).fillna(0).values
    scored['pop_good'] = scored['tid'].map(pop_good).fillna(0).values
    return scored

val_scored = add_popularity_scores(train_part_base, val_part)

pop_results = {}
for col in ['pop_sum', 'pop_mean', 'pop_count', 'pop_good']:
    score = evaluate_ndcg(val_scored, score_col=col)
    pop_results[col] = score
    print(f"{col:10s} NDCG@10: {score:.5f}")

best_pop_col = max(pop_results, key=pop_results.get)
ndcg_popularity = pop_results[best_pop_col]
print(f"\nBest popularity: {best_pop_col}, NDCG@10={ndcg_popularity:.5f}")

pop_sum    NDCG@10: 0.63820
pop_mean   NDCG@10: 0.66001
pop_count  NDCG@10: 0.57677
pop_good   NDCG@10: 0.64724

Best popularity: pop_mean, NDCG@10=0.66001


## 6. ALS (Alternating Least Squares)

In [21]:
import implicit

print(f"implicit version: {implicit.__version__}")


def train_als(params, interactions):
    user_item_train = build_user_item(interactions)
    model = implicit.als.AlternatingLeastSquares(
        factors=params['factors'],
        iterations=params['iterations'],
        regularization=params['regularization'],
        random_state=42,
        use_gpu=False,
    )
    model.fit(user_item_train, show_progress=False)
    return model


def evaluate_als_params(params):
    interactions = add_confidence(
        train_part_base,
        alpha=params['alpha'],
        min_time=params['min_time'],
        formula=params['formula'],
    )
    model = train_als(params, interactions)
    pred = als_candidate_scores(model, val_part)
    val_part_tmp = val_part.copy()
    val_part_tmp['pred'] = pred
    return evaluate_ndcg(val_part_tmp, score_col='pred'), model

# Быстрый sanity-check ALS.
default_params = {
    'alpha': 40,
    'min_time': 0.0,
    'formula': 'max_log_count',
    'factors': 128,
    'regularization': 0.01,
    'iterations': 50,
}
ndcg_als, als_model = evaluate_als_params(default_params)
print(f"Default ALS NDCG@10: {ndcg_als:.5f}")
print(f"Прирост vs best popularity: {ndcg_als - ndcg_popularity:+.5f}")

implicit version: 0.7.2
Default ALS NDCG@10: 0.72193
Прирост vs best popularity: +0.06191


In [22]:
# Оцениваем ALS на val
user_factors  = als_model.user_factors   # (n_users, factors)
item_factors  = als_model.item_factors   # (n_items, factors)

# Скор = dot product user_vec · item_vec
val_uids = val_part['uid'].values
val_tids = val_part['tid'].values

val_part['pred'] = np.sum(
    user_factors[val_uids] * item_factors[val_tids], axis=1
)

ndcg_als = evaluate_ndcg(val_part, score_col='pred')
print(f"ALS NDCG@10: {ndcg_als:.4f}")
print(f"Прирост vs Popularity: {ndcg_als - ndcg_popularity:+.4f}")

ALS NDCG@10: 0.7219
Прирост vs Popularity: +0.0619


## 7. ALS — подбор гиперпараметров

In [23]:
# Быстрый режим: фиксируемся на лучших настройках из уже запущенного grid search.
# Лучший результат в твоем прогоне: alpha=20, min_time=0.1, formula=mix, factors=64, reg=0.05.
# Оставляем только несколько близких конфигураций для ансамбля, чтобы не ждать 72 запуска.

param_grid = [
    {'alpha': 20, 'min_time': 0.10, 'formula': 'mix', 'factors': 64, 'regularization': 0.05, 'iterations': 50},
    {'alpha': 20, 'min_time': 0.10, 'formula': 'mix', 'factors': 64, 'regularization': 0.01, 'iterations': 50},
    {'alpha': 20, 'min_time': 0.10, 'formula': 'max_log_count', 'factors': 64, 'regularization': 0.05, 'iterations': 50},
    {'alpha': 20, 'min_time': 0.10, 'formula': 'max_log_count', 'factors': 64, 'regularization': 0.01, 'iterations': 50},
    {'alpha': 20, 'min_time': 0.00, 'formula': 'max_log_count', 'factors': 64, 'regularization': 0.05, 'iterations': 50},
    {'alpha': 20, 'min_time': 0.00, 'formula': 'mix', 'factors': 64, 'regularization': 0.05, 'iterations': 50},
]

results = []
models_for_ensemble = []

for i, params in enumerate(param_grid, start=1):
    score, model = evaluate_als_params(params)
    row = dict(params)
    row['ndcg'] = score
    results.append(row)

    models_for_ensemble.append((score, params, model))
    models_for_ensemble = sorted(models_for_ensemble, key=lambda x: x[0], reverse=True)[:5]

    print(
        f"[{i:03d}/{len(param_grid)}] "
        f"NDCG={score:.5f} alpha={params['alpha']} min_time={params['min_time']} "
        f"formula={params['formula']} factors={params['factors']} reg={params['regularization']}"
    )

results_df = pd.DataFrame(results).sort_values('ndcg', ascending=False).reset_index(drop=True)
best = results_df.iloc[0].to_dict()
print("\nTop ALS params:")
display(results_df)

[001/6] NDCG=0.75304 alpha=20 min_time=0.1 formula=mix factors=64 reg=0.05
[002/6] NDCG=0.75286 alpha=20 min_time=0.1 formula=mix factors=64 reg=0.01
[003/6] NDCG=0.75265 alpha=20 min_time=0.1 formula=max_log_count factors=64 reg=0.05
[004/6] NDCG=0.75258 alpha=20 min_time=0.1 formula=max_log_count factors=64 reg=0.01
[005/6] NDCG=0.74734 alpha=20 min_time=0.0 formula=max_log_count factors=64 reg=0.05
[006/6] NDCG=0.74731 alpha=20 min_time=0.0 formula=mix factors=64 reg=0.05

Top ALS params:


,alpha,min_time,formula,factors,regularization,iterations,ndcg
0,20,0.1,mix,64,0.05,50,0.753038
1,20,0.1,mix,64,0.01,50,0.752861
2,20,0.1,max_log_count,64,0.05,50,0.752654
3,20,0.1,max_log_count,64,0.01,50,0.752584
4,20,0.0,max_log_count,64,0.05,50,0.747339
5,20,0.0,mix,64,0.05,50,0.747306


## 8. Финальная модель на всём train + генерация submission

In [24]:
# Подбираем ансамбль на validation: несколько лучших ALS + лучшая popularity.
top_models = models_for_ensemble[:3]
blend_val = val_part.copy()

for model_idx, (_, params, model) in enumerate(top_models):
    raw = als_candidate_scores(model, val_part)
    blend_val[f'als_{model_idx}'] = normalize_per_user(val_part, raw, f'als_{model_idx}')
    print(f"ALS {model_idx}: params={params}")

blend_val['pop'] = normalize_per_user(val_part, val_scored[best_pop_col].values, 'pop')

# Сетка весов для 3 ALS + popularity. Веса суммируются в 1.
blend_results = []
for w0 in np.linspace(0, 1, 11):
    for w1 in np.linspace(0, 1 - w0, 11):
        for w2 in np.linspace(0, 1 - w0 - w1, 11):
            w_pop = 1 - w0 - w1 - w2
            if w_pop < -1e-9:
                continue
            blend_val['pred'] = (
                w0 * blend_val['als_0'] +
                w1 * blend_val.get('als_1', 0) +
                w2 * blend_val.get('als_2', 0) +
                w_pop * blend_val['pop']
            )
            score = evaluate_ndcg(blend_val, score_col='pred')
            blend_results.append({'w0': w0, 'w1': w1, 'w2': w2, 'w_pop': w_pop, 'ndcg': score})

blend_results_df = pd.DataFrame(blend_results).sort_values('ndcg', ascending=False).reset_index(drop=True)
best_blend = blend_results_df.iloc[0].to_dict()
print("\nBest blend:")
display(blend_results_df.head(10))

ALS 0: params={'alpha': 20, 'min_time': 0.1, 'formula': 'mix', 'factors': 64, 'regularization': 0.05, 'iterations': 50}
ALS 1: params={'alpha': 20, 'min_time': 0.1, 'formula': 'mix', 'factors': 64, 'regularization': 0.01, 'iterations': 50}
ALS 2: params={'alpha': 20, 'min_time': 0.1, 'formula': 'max_log_count', 'factors': 64, 'regularization': 0.05, 'iterations': 50}

Best blend:


,w0,w1,w2,w_pop,ndcg
0,0.6,0.12,0.028,0.252,0.764140
1,0.4,0.24,0.108,0.252,0.764138
2,0.5,0.00,0.250,0.250,0.764132
3,0.3,0.28,0.168,0.252,0.764122
4,0.4,0.18,0.168,0.252,0.764121
5,0.3,0.07,0.378,0.252,0.764103
6,0.5,0.15,0.105,0.245,0.764101
7,0.6,0.04,0.108,0.252,0.764087
8,0.5,0.25,0.000,0.250,0.764080
9,0.6,0.08,0.064,0.256,0.764073


## 8.1 Learning-to-rank поверх ALS


In [ ]:
# Новая технология: GBDT ranker/regressor поверх ALS + popularity + user/item/context статистик.
# В окружении есть sklearn, поэтому используем HistGradientBoostingRegressor.
from sklearn.ensemble import HistGradientBoostingRegressor

MAX_RANKER_ROWS = 500_000
RANKER_RANDOM_STATE = 42


def unpack_model_entry(entry):
    """top_models хранит (score, params, model), final_models хранит (params, model)."""
    if len(entry) == 3:
        _, params, model = entry
    else:
        params, model = entry
    return params, model


def with_track_meta(df):
    return df.join(track_meta_ltr, on='tid')


def build_feature_stats(base_df):
    base = with_track_meta(base_df)

    user_stats = base.groupby('uid').agg(
        user_tracks=('tid', 'count'),
        user_time_mean=('time_mean', 'mean'),
        user_time_sum=('time_sum', 'sum'),
        user_good_rate=('time_max', lambda x: (x >= 0.7).mean()),
        user_bad_rate=('time_max', lambda x: (x < 0.1).mean()),
        user_year_mean=('year', 'mean'),
        user_artist_fans_mean=('artist_fans', 'mean'),
    )
    item_stats = base.groupby('tid').agg(
        item_users=('uid', 'count'),
        item_time_mean=('time_mean', 'mean'),
        item_time_sum=('time_sum', 'sum'),
        item_good_rate=('time_max', lambda x: (x >= 0.7).mean()),
        item_bad_rate=('time_max', lambda x: (x < 0.1).mean()),
    )
    artist_stats = base.groupby('artist_id').agg(
        artist_users=('uid', 'count'),
        artist_time_mean=('time_mean', 'mean'),
        artist_time_sum=('time_sum', 'sum'),
        artist_good_rate=('time_max', lambda x: (x >= 0.7).mean()),
        artist_bad_rate=('time_max', lambda x: (x < 0.1).mean()),
    )
    genre_stats = base.groupby('artist_genre_code').agg(
        genre_users=('uid', 'count'),
        genre_time_mean=('time_mean', 'mean'),
        genre_time_sum=('time_sum', 'sum'),
        genre_good_rate=('time_max', lambda x: (x >= 0.7).mean()),
    )
    country_stats = base.groupby('artist_country_code').agg(
        country_users=('uid', 'count'),
        country_time_mean=('time_mean', 'mean'),
        country_time_sum=('time_sum', 'sum'),
        country_good_rate=('time_max', lambda x: (x >= 0.7).mean()),
    )
    return user_stats, item_stats, artist_stats, genre_stats, country_stats


def build_cross_stats(base_df):
    base = with_track_meta(base_df)

    user_artist = base.groupby(['uid', 'artist_id']).agg(
        user_artist_count=('tid', 'count'),
        user_artist_time_mean=('time_mean', 'mean'),
        user_artist_time_sum=('time_sum', 'sum'),
        user_artist_good_rate=('time_max', lambda x: (x >= 0.7).mean()),
    )
    user_genre = base.groupby(['uid', 'artist_genre_code']).agg(
        user_genre_count=('tid', 'count'),
        user_genre_time_mean=('time_mean', 'mean'),
        user_genre_time_sum=('time_sum', 'sum'),
        user_genre_good_rate=('time_max', lambda x: (x >= 0.7).mean()),
    )
    user_country = base.groupby(['uid', 'artist_country_code']).agg(
        user_country_count=('tid', 'count'),
        user_country_time_mean=('time_mean', 'mean'),
        user_country_time_sum=('time_sum', 'sum'),
    )
    return user_artist, user_genre, user_country


def join_multiindex_features(features, scored, stats, keys):
    key_frame = scored[keys]
    joined = key_frame.join(stats, on=keys)
    for col in stats.columns:
        features[col] = joined[col].values


def make_ltr_features(base_df, eval_df, model_entries):
    scored = add_popularity_scores(base_df, eval_df).copy()
    scored = with_track_meta(scored)
    user_stats, item_stats, artist_stats, genre_stats, country_stats = build_feature_stats(base_df)
    user_artist, user_genre, user_country = build_cross_stats(base_df)

    features = pd.DataFrame(index=scored.index)
    features['uid'] = scored['uid'].astype(np.int32)
    features['tid'] = scored['tid'].astype(np.int32)

    meta_cols = [
        'year', 'artist_fans', 'artist_id', 'artist_code', 'artist_country_code',
        'artist_genre_code', 'artist_genres_str_code', 'genres_str_code', 'mood_str_code'
    ]
    for col in meta_cols:
        features[col] = scored[col].fillna(0).astype(np.float32)

    features['track_age'] = (2026 - scored['year'].fillna(2026)).astype(np.float32)
    features['log_artist_fans'] = np.log1p(scored['artist_fans'].fillna(0)).astype(np.float32)

    for col in ['pop_sum', 'pop_mean', 'pop_count', 'pop_good']:
        features[col] = scored[col].astype(np.float32)
        features[f'{col}_rank'] = scored.groupby('uid')[col].rank(pct=True).astype(np.float32)

    features = features.join(user_stats, on='uid')
    features = features.join(item_stats, on='tid')
    features = features.join(artist_stats, on='artist_id')
    features = features.join(genre_stats, on='artist_genre_code')
    features = features.join(country_stats, on='artist_country_code')

    join_multiindex_features(features, scored, user_artist, ['uid', 'artist_id'])
    join_multiindex_features(features, scored, user_genre, ['uid', 'artist_genre_code'])
    join_multiindex_features(features, scored, user_country, ['uid', 'artist_country_code'])

    # Relative/preference features: насколько кандидат близок к историческим вкусам пользователя.
    features['year_diff_user_mean'] = (features['year'] - features['user_year_mean']).astype(np.float32)
    features['fans_diff_user_mean'] = (features['artist_fans'] - features['user_artist_fans_mean']).astype(np.float32)
    features['user_artist_share'] = features['user_artist_count'] / (features['user_tracks'] + 1e-6)
    features['user_genre_share'] = features['user_genre_count'] / (features['user_tracks'] + 1e-6)
    features['user_country_share'] = features['user_country_count'] / (features['user_tracks'] + 1e-6)

    blend_score = np.zeros(len(scored), dtype=np.float32)
    blend_weights = [best_blend.get('w0', 0.0), best_blend.get('w1', 0.0), best_blend.get('w2', 0.0)]

    for model_idx, entry in enumerate(model_entries[:3]):
        _, model = unpack_model_entry(entry)
        raw = als_candidate_scores(model, scored)
        norm = normalize_per_user(scored, raw, f'als_{model_idx}')
        features[f'als_{model_idx}_raw'] = raw.astype(np.float32)
        features[f'als_{model_idx}_norm'] = norm.astype(np.float32)
        features[f'als_{model_idx}_rank'] = pd.Series(raw, index=scored.index).groupby(scored['uid']).rank(pct=True).astype(np.float32)
        blend_score += float(blend_weights[model_idx]) * norm

    pop_norm = normalize_per_user(scored, scored[best_pop_col].values, 'pop')
    blend_score += float(best_blend.get('w_pop', 0.0)) * pop_norm
    features['best_blend_score'] = blend_score

    # Еще несколько rank-фичей после появления контентных признаков.
    for col in ['user_artist_share', 'user_genre_share', 'artist_time_mean', 'genre_time_mean', 'track_age']:
        features[f'{col}_rank'] = features[col].groupby(scored['uid']).rank(pct=True).astype(np.float32)

    return features.replace([np.inf, -np.inf], np.nan).fillna(0.0)


ranker_train = train_part_base.copy()
if len(ranker_train) > MAX_RANKER_ROWS:
    ranker_train = ranker_train.sample(MAX_RANKER_ROWS, random_state=RANKER_RANDOM_STATE)

X_ranker_train = make_ltr_features(train_part_base, ranker_train, top_models)
y_ranker_train = ranker_train['time'].values.astype(np.float32)
X_ranker_val = make_ltr_features(train_part_base, val_part, top_models)

ranker_model = HistGradientBoostingRegressor(
    loss='squared_error',
    learning_rate=0.040,
    max_iter=700,
    max_leaf_nodes=63,
    min_samples_leaf=25,
    l2_regularization=0.03,
    random_state=RANKER_RANDOM_STATE,
)
ranker_model.fit(X_ranker_train, y_ranker_train)

val_ranker = val_part.copy()
val_ranker['pred'] = ranker_model.predict(X_ranker_val)
ndcg_ranker = evaluate_ndcg(val_ranker, score_col='pred')

print(f"GBDT ranker NDCG@10: {ndcg_ranker:.5f}")
print(f"Delta vs ALS blend : {ndcg_ranker - best_blend['ndcg']:+.5f}")
print(f"Ranker train rows  : {len(ranker_train):,}")
print(f"Features           : {len(X_ranker_train.columns)}")
print(list(X_ranker_train.columns))

## 8.2 Torch MLP ranker


In [ ]:
# Neural ranker на PyTorch поверх LTR-признаков.
# Если доступна GPU, torch сам возьмет cuda; на Mac может взять mps.
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    from sklearn.preprocessing import StandardScaler

    TORCH_AVAILABLE = True
except ModuleNotFoundError as e:
    TORCH_AVAILABLE = False
    ndcg_torch = -np.inf
    print(f"PyTorch не установлен в этом окружении, пропускаю neural ranker: {e}")

if TORCH_AVAILABLE:
    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
        device = torch.device('mps')
    else:
        device = torch.device('cpu')

    torch.manual_seed(42)
    np.random.seed(42)

    # Используем уже построенные признаки из GBDT-блока.
    torch_scaler = StandardScaler()
    X_train_np = torch_scaler.fit_transform(X_ranker_train).astype(np.float32)
    y_train_np = y_ranker_train.astype(np.float32)
    X_val_np = torch_scaler.transform(X_ranker_val).astype(np.float32)

    train_ds = TensorDataset(
        torch.from_numpy(X_train_np),
        torch.from_numpy(y_train_np[:, None]),
    )
    train_loader = DataLoader(train_ds, batch_size=8192, shuffle=True, num_workers=0)

    class TorchRanker(nn.Module):
        def __init__(self, n_features):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(n_features, 256),
                nn.BatchNorm1d(256),
                nn.SiLU(),
                nn.Dropout(0.15),
                nn.Linear(256, 128),
                nn.BatchNorm1d(128),
                nn.SiLU(),
                nn.Dropout(0.10),
                nn.Linear(128, 64),
                nn.SiLU(),
                nn.Linear(64, 1),
            )

        def forward(self, x):
            return self.net(x)

    torch_mlp = TorchRanker(X_train_np.shape[1]).to(device)
    optimizer = torch.optim.AdamW(torch_mlp.parameters(), lr=2e-3, weight_decay=2e-4)
    loss_fn = nn.SmoothL1Loss(beta=0.08)

    best_torch_state = None
    best_torch_loss = np.inf
    patience = 4
    bad_epochs = 0

    # Небольшая, но сильная сеть; обычно 10-20 эпох хватает для табличных LTR-фичей.
    for epoch in range(1, 31):
        torch_mlp.train()
        total_loss = 0.0
        total_rows = 0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad(set_to_none=True)
            pred = torch_mlp(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(torch_mlp.parameters(), 3.0)
            optimizer.step()

            total_loss += float(loss.detach().cpu()) * len(xb)
            total_rows += len(xb)

        mean_loss = total_loss / max(total_rows, 1)
        if mean_loss < best_torch_loss - 1e-5:
            best_torch_loss = mean_loss
            best_torch_state = {k: v.detach().cpu().clone() for k, v in torch_mlp.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1

        print(f"epoch={epoch:02d} loss={mean_loss:.6f} device={device}")
        if bad_epochs >= patience:
            break

    if best_torch_state is not None:
        torch_mlp.load_state_dict(best_torch_state)
    torch_mlp.eval()

    with torch.no_grad():
        val_pred = []
        val_tensor = torch.from_numpy(X_val_np)
        for start in range(0, len(val_tensor), 65536):
            batch = val_tensor[start:start + 65536].to(device)
            val_pred.append(torch_mlp(batch).detach().cpu().numpy().ravel())
        val_pred = np.concatenate(val_pred)

    val_torch = val_part.copy()
    val_torch['pred'] = val_pred
    ndcg_torch = evaluate_ndcg(val_torch, score_col='pred')

    print(f"Torch MLP NDCG@10: {ndcg_torch:.5f}")
    print(f"Delta vs GBDT     : {ndcg_torch - ndcg_ranker:+.5f}")
    print(f"Delta vs blend    : {ndcg_torch - best_blend['ndcg']:+.5f}")

## 8.3 Stacking blend


In [ ]:
# Stacking поверх лучших скорингов: несколько GBDT + бленд GBDT/Torch/ALS по NDCG.
# Это дешевле и надежнее, чем пытаться обучать Transformer без явной последовательности событий.

gbdt_configs = [
    dict(learning_rate=0.035, max_iter=650, max_leaf_nodes=31, min_samples_leaf=25, l2_regularization=0.02, random_state=11),
    dict(learning_rate=0.045, max_iter=500, max_leaf_nodes=31, min_samples_leaf=30, l2_regularization=0.03, random_state=42),
    dict(learning_rate=0.030, max_iter=750, max_leaf_nodes=63, min_samples_leaf=35, l2_regularization=0.05, random_state=77),
    dict(learning_rate=0.055, max_iter=420, max_leaf_nodes=15, min_samples_leaf=20, l2_regularization=0.01, random_state=123),
]

gbdt_models = []
gbdt_val_preds = []

for i, cfg in enumerate(gbdt_configs):
    model = HistGradientBoostingRegressor(loss='squared_error', **cfg)
    model.fit(X_ranker_train, y_ranker_train)
    pred = model.predict(X_ranker_val)

    tmp = val_part.copy()
    tmp['pred'] = pred
    score = evaluate_ndcg(tmp, score_col='pred')
    gbdt_models.append(model)
    gbdt_val_preds.append(pred)
    print(f"GBDT[{i}] NDCG@10={score:.5f} cfg={cfg}")

gbdt_ensemble_pred = np.mean(gbdt_val_preds, axis=0)
val_gbdt_ens = val_part.copy()
val_gbdt_ens['pred'] = gbdt_ensemble_pred
ndcg_gbdt_ensemble = evaluate_ndcg(val_gbdt_ens, score_col='pred')
print(f"GBDT ensemble NDCG@10: {ndcg_gbdt_ensemble:.5f}")

# Блендим уже готовые скоринги. Нормализация внутри пользователя важна: масштабы моделей разные.
stack_val = val_part.copy()
stack_val['als_blend'] = normalize_per_user(val_part, blend_val['pred'].values, 'als_blend')
stack_val['gbdt'] = normalize_per_user(val_part, val_ranker['pred'].values, 'gbdt')
stack_val['gbdt_ensemble'] = normalize_per_user(val_part, gbdt_ensemble_pred, 'gbdt_ensemble')

if 'val_torch' in globals() and 'ndcg_torch' in globals() and np.isfinite(ndcg_torch):
    stack_val['torch_mlp'] = normalize_per_user(val_part, val_torch['pred'].values, 'torch_mlp')
else:
    stack_val['torch_mlp'] = 0.0

stack_results = []
for w_gbdt_ens in np.linspace(0, 1, 21):
    for w_gbdt in np.linspace(0, 1 - w_gbdt_ens, 21):
        for w_torch in np.linspace(0, 1 - w_gbdt_ens - w_gbdt, 21):
            w_als = 1 - w_gbdt_ens - w_gbdt - w_torch
            if w_als < -1e-9:
                continue
            stack_val['pred'] = (
                w_gbdt_ens * stack_val['gbdt_ensemble'] +
                w_gbdt * stack_val['gbdt'] +
                w_torch * stack_val['torch_mlp'] +
                w_als * stack_val['als_blend']
            )
            score = evaluate_ndcg(stack_val, score_col='pred')
            stack_results.append({
                'w_gbdt_ensemble': w_gbdt_ens,
                'w_gbdt': w_gbdt,
                'w_torch': w_torch,
                'w_als': w_als,
                'ndcg': score,
            })

stack_results_df = pd.DataFrame(stack_results).sort_values('ndcg', ascending=False).reset_index(drop=True)
best_stack = stack_results_df.iloc[0].to_dict()
ndcg_stack = best_stack['ndcg']

print("\nBest stack:")
display(stack_results_df.head(10))

In [ ]:
# Обучаем финальные модели на всем train с параметрами лучших validation-моделей.
final_models = []
for _, params, _ in top_models:
    full_interactions = add_confidence(
        train_agg,
        alpha=params['alpha'],
        min_time=params['min_time'],
        formula=params['formula'],
    )
    final_model = train_als(params, full_interactions)
    final_models.append((params, final_model))

# Финальные popularity-скоринги считаем по всему train.
test_scored = add_popularity_scores(train_agg, test)
submission_scores = np.zeros(len(test), dtype=np.float32)
weights = [best_blend['w0'], best_blend['w1'], best_blend['w2']]

for model_idx, (params, model) in enumerate(final_models):
    raw = als_candidate_scores(model, test)
    norm = normalize_per_user(test, raw, f'als_{model_idx}')
    submission_scores += float(weights[model_idx]) * norm

pop_norm = normalize_per_user(test, test_scored[best_pop_col].values, 'pop')
submission_scores += float(best_blend['w_pop']) * pop_norm

# Сохраняем все доступные варианты, чтобы можно было отправлять/сравнивать их отдельно.
saved_submissions = {}

def save_submission(score_values, filename):
    out = test[['user', 'track']].copy()
    out['score'] = np.asarray(score_values)
    path = DATA_DIR / filename
    out.to_csv(path, index=False)
    saved_submissions[filename] = path
    print(f"Сохранено: {path}")
    return path

save_submission(submission_scores, "submission_als_ensemble.csv")

need_ltr_features = any(name in globals() for name in [
    'ranker_model', 'gbdt_models', 'torch_mlp', 'best_stack'
])
if need_ltr_features:
    X_test_ranker = make_ltr_features(train_agg, test, final_models)

if 'ranker_model' in globals():
    gbdt_test = ranker_model.predict(X_test_ranker)
    save_submission(gbdt_test, "submission_ltr_gbdt.csv")

if 'gbdt_models' in globals():
    gbdt_ens_test = np.mean([model.predict(X_test_ranker) for model in gbdt_models], axis=0)
    save_submission(gbdt_ens_test, "submission_gbdt_ensemble.csv")

if 'torch_mlp' in globals() and 'torch_scaler' in globals():
    X_test_np = torch_scaler.transform(X_test_ranker).astype(np.float32)
    torch_mlp.eval()
    preds = []
    with torch.no_grad():
        test_tensor = torch.from_numpy(X_test_np)
        for start in range(0, len(test_tensor), 65536):
            batch = test_tensor[start:start + 65536].to(device)
            preds.append(torch_mlp(batch).detach().cpu().numpy().ravel())
    torch_test = np.concatenate(preds)
    save_submission(torch_test, "submission_torch_mlp.csv")

if 'best_stack' in globals():
    stack_test_score = np.zeros(len(test), dtype=np.float32)

    if 'gbdt_ens_test' in globals():
        stack_test_score += float(best_stack['w_gbdt_ensemble']) * normalize_per_user(test, gbdt_ens_test, 'gbdt_ensemble')
    if 'gbdt_test' in globals():
        stack_test_score += float(best_stack['w_gbdt']) * normalize_per_user(test, gbdt_test, 'gbdt')
    if 'torch_test' in globals():
        stack_test_score += float(best_stack['w_torch']) * normalize_per_user(test, torch_test, 'torch_mlp')

    stack_test_score += float(best_stack['w_als']) * normalize_per_user(test, submission_scores, 'als_blend')
    save_submission(stack_test_score, "submission_stack_ltr.csv")

candidates = {'submission_als_ensemble.csv': best_blend['ndcg']}
if 'ndcg_ranker' in globals():
    candidates['submission_ltr_gbdt.csv'] = ndcg_ranker
if 'ndcg_gbdt_ensemble' in globals():
    candidates['submission_gbdt_ensemble.csv'] = ndcg_gbdt_ensemble
if 'ndcg_stack' in globals():
    candidates['submission_stack_ltr.csv'] = ndcg_stack
if 'ndcg_torch' in globals() and np.isfinite(ndcg_torch):
    candidates['submission_torch_mlp.csv'] = ndcg_torch

best_submission_name = max(candidates, key=candidates.get)
print("\nValidation leaderboard:")
for name, score in sorted(candidates.items(), key=lambda x: x[1], reverse=True):
    print(f"{name:32s} NDCG@10={score:.5f}")

print(f"\nЛучший по validation: {best_submission_name}")
saved_submissions

## 9. Итоги

In [ ]:
print("=" * 60)
print("РЕЗУЛЬТАТЫ ВАЛИДАЦИИ (NDCG@10)")
print("=" * 60)
print(f"Best popularity      : {ndcg_popularity:.5f} ({best_pop_col})")
print(f"Default ALS          : {ndcg_als:.5f}")
print(f"Best ALS grid search : {best['ndcg']:.5f}")
print(f"Best ALS params      : { {k: best[k] for k in ['alpha', 'min_time', 'formula', 'factors', 'regularization', 'iterations']} }")
print(f"Best blend           : {best_blend['ndcg']:.5f}")
if 'ndcg_ranker' in globals():
    print(f"GBDT ranker          : {ndcg_ranker:.5f}")
if 'ndcg_torch' in globals() and np.isfinite(ndcg_torch):
    print(f"Torch MLP            : {ndcg_torch:.5f}")
if 'ndcg_gbdt_ensemble' in globals():
    print(f"GBDT ensemble        : {ndcg_gbdt_ensemble:.5f}")
if 'ndcg_stack' in globals():
    print(f"Stack blend          : {ndcg_stack:.5f}")
    print(f"Stack weights        : {best_stack}")
print(f"Blend weights        : ALS0={best_blend['w0']:.2f}, ALS1={best_blend['w1']:.2f}, ALS2={best_blend['w2']:.2f}, pop={best_blend['w_pop']:.2f}")
print()
print("Файл для сабмита выбирается автоматически: submission_stack_ltr.csv / submission_gbdt_ensemble.csv / submission_ltr_gbdt.csv / submission_torch_mlp.csv / submission_als_ensemble.csv")